# 多項プロビット（MNP）の選択確率の導出

本ノートブックでは、多項プロビットモデルにおける選択確率
$P_{ij}=P(y_i=j)$ を、効用最大化の定義から導出する。
最後に 3 択の簡単なモンテカルロ確認を行う。


## 1. モデル設定

個体 $i$、選択肢 $j\in\{1,\ldots,J\}$ に対して効用を

$$
U_{ij}=V_{ij}+\varepsilon_{ij}
$$

と置く。ここで $V_{ij}$ は決定的効用、$\varepsilon_{ij}$ は誤差項である。
多項プロビットでは

$$
\boldsymbol{\varepsilon}_i=(\varepsilon_{i1},\ldots,\varepsilon_{iJ})^\top \sim \mathcal{N}(\mathbf{0},\boldsymbol{\Sigma})
$$

を仮定する。


## 2. 選択事象の定義

個体 $i$ が選択肢 $j$ を選ぶ事象は

$$
y_i=j \iff U_{ij}\ge U_{ik},\ \forall k\neq j
$$

である。したがって選択確率は

$$
\begin{aligned}
P_{ij}
&=P(y_i=j)\\
&=P\big(V_{ij}+\varepsilon_{ij}\ge V_{ik}+\varepsilon_{ik},\ \forall k\neq j\big).
\end{aligned}
$$


## 3. 差分への変換

上式の不等式を並べ替えると、各 $k\neq j$ について

$$
\varepsilon_{ik}-\varepsilon_{ij} \le V_{ij}-V_{ik}
$$

となる。そこで差分ベクトルを

$$
d_i^{(j)}:=
\begin{bmatrix}
\varepsilon_{i1}-\varepsilon_{ij}\\
\vdots\\
\varepsilon_{i,j-1}-\varepsilon_{ij}\\
\varepsilon_{i,j+1}-\varepsilon_{ij}\\
\vdots\\
\varepsilon_{iJ}-\varepsilon_{ij}
\end{bmatrix},
\qquad
\delta_i^{(j)}:=
\begin{bmatrix}
V_{ij}-V_{i1}\\
\vdots\\
V_{ij}-V_{i,j-1}\\
V_{ij}-V_{i,j+1}\\
\vdots\\
V_{ij}-V_{iJ}
\end{bmatrix}
$$

と定義すると、

$$
P_{ij}=P\big(d_i^{(j)}\le \delta_i^{(j)}\big)
$$

（成分ごとの不等号）と書ける。


## 4. 多変量正規積分としての表現

$d_i^{(j)}$ は $\boldsymbol{\varepsilon}_i$ の線形変換であるため、

$$
d_i^{(j)} \sim \mathcal{N}(\mathbf{0},\boldsymbol{\Omega}_j)
$$

となる。ここで $\boldsymbol{\Omega}_j$ は差分の分散共分散行列である。
したがって選択確率は $(J-1)$ 次元正規CDFで

$$
\boxed{
P_{ij}=\int_{-\infty}^{\delta_{i1}^{(j)}}\cdots\int_{-\infty}^{\delta_{i,J-1}^{(j)}}
\phi_{J-1}(t;\mathbf{0},\boldsymbol{\Omega}_j)\,dt
}
$$

と表される。ここで $\phi_{J-1}(\cdot;\mathbf{0},\boldsymbol{\Omega}_j)$ は
$(J-1)$ 次元正規密度である。

この式は一般に閉形式を持たず、数値積分またはシミュレーション積分で評価する。


## 5. 識別のための正規化

効用は位置と尺度に関して不識別であるため、推定時には正規化が必要である。
代表的には次の制約を置く。

- 位置の正規化: ある基準選択肢の切片を 0 に固定する。
- 尺度の正規化: 誤差差分の分散を 1 に固定する（または同等の制約を置く）。


In [ ]:
# 3択の簡単なモンテカルロ確認
import numpy as np

rng = np.random.default_rng(42)
n = 300_000

# 決定的効用（例）
V = np.array([0.6, 0.4, 0.2])

# 誤差の共分散（対角以外を入れてMNPらしさを出す）
Sigma = np.array(
    [
        [1.0, 0.4, 0.1],
        [0.4, 1.0, 0.3],
        [0.1, 0.3, 1.0],
    ]
)

eps = rng.multivariate_normal(mean=np.zeros(3), cov=Sigma, size=n)
U = V + eps
choice = np.argmax(U, axis=1)

probs = np.bincount(choice, minlength=3) / n
for j, p in enumerate(probs, start=1):
    print(f"P(y={j}) ≈ {p:.4f}")

print("sum =", probs.sum())

## 6. まとめ

多項プロビットの選択確率は、効用差の不等式系を満たす確率として定義され、
最終的に $(J-1)$ 次元正規CDF（多変量正規積分）として表される。
このため MNL と比べて柔軟性は高いが、計算コストは大きい。
